# Full RAG + Uncertainty Estimation Pipeline  
### Using Qwen2.5-7B-Instruct (bf16), SAFE-Lite, MARS (TruthTorch or fallback), ECC  
### IR Project — Arnes HPC Jupyter Environment


In [1]:
import os

# HuggingFace cache redirection (MUST be FIRST cell)
os.environ["HF_HOME"] = "/d/hpc/projects/FRI/ma76193/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_transformers"

# Create folders if missing
for path in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    os.makedirs(path, exist_ok=True)

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE =", os.environ["TRANSFORMERS_CACHE"])


HF_HOME = /d/hpc/projects/FRI/ma76193/hf_home
HF_DATASETS_CACHE = /d/hpc/projects/FRI/ma76193/hf_datasets
TRANSFORMERS_CACHE = /d/hpc/projects/FRI/ma76193/hf_transformers


In [ ]:
import os
import sys
import json
import time
import random
import logging
import threading
import subprocess
from pathlib import Path
from datetime import datetime

JAVA = "/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12"

os.environ["JAVA_HOME"] = JAVA
os.environ["JVM_PATH"] = f"{JAVA}/lib/server/libjvm.so"
os.environ["LD_LIBRARY_PATH"] = f"{JAVA}/lib/server:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PATH"] = f"{JAVA}/bin:" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
!java -version
!find $JAVA_HOME -name "libjvm.so"

import torch

# Logging
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "full_notebook.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
              logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("nb")

def banner(msg):
    line = "=" * 80
    log.info(line)
    log.info(f"*** {msg} ***")
    log.info(line)


JAVA_HOME = /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12
openjdk version "21.0.1" 2023-10-17 LTS
OpenJDK Runtime Environment Temurin-21.0.1+12 (build 21.0.1+12-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.1+12 (build 21.0.1+12-LTS, mixed mode, sharing)
/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so


In [ ]:
!wget -O /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/openjdk21.tar.gz \
    https://github.com/adoptium/temurin21-binaries/releases/download/jdk-21.0.1%2B12/OpenJDK21U-jdk_x64_linux_hotspot_21.0.1_12.tar.gz
!tar -xzf /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/openjdk21.tar.gz \
    -C /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/
!ls /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311


In [ ]:
# Heartbeat to show notebook is alive during long retrieval/index steps
_stop_hb = threading.Event()

def _heartbeat(period=30):
    while not _stop_hb.is_set():
        log.info("[HEARTBEAT] Notebook run alive...")
        time.sleep(period)

hb = threading.Thread(target=_heartbeat, daemon=True)
hb.start()



In [ ]:
!pip install ipywidgets


In [ ]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Xms1g -Xmx8g")

log.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        log.info(f"GPU[{i}] {torch.cuda.get_device_name(i)}")


In [ ]:
FULL_SHARDS_DIR = Path("data/wiki18/shards_full")
FULL_INDEX_DIR  = Path("index/bm25_full")

RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)

RETR_DIR = RUNS / "retrieval"
ANS_DIR  = RUNS / "answers"
UE_DIR   = RUNS / "ue"
REPORTS_DIR = Path("reports")

for p in [RETR_DIR, ANS_DIR, UE_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

banner("Directories prepared")


## Download wiki-18 Dataset (idempotent)
Checks if already downloaded in HF cache.  
If not present → downloads only `.jsonl.gz`.


In [ ]:
from huggingface_hub import snapshot_download

HF_DATA_REPO = "PeterJinGo/wiki-18-corpus"

def download_wiki18():
    banner("DOWNLOAD OR USE CACHED WIKI-18")
    path = snapshot_download(
        repo_id=HF_DATA_REPO,
        repo_type="dataset",
        allow_patterns=["*.jsonl.gz"],
        local_files_only=False
    )
    snapshot = Path(path)
    candidates = list(snapshot.rglob("*.jsonl.gz"))
    if not candidates:
        raise RuntimeError("No wiki18 .jsonl.gz found")
    return candidates[0]

wiki_gz = download_wiki18()
log.info(f"wiki gz: {wiki_gz}")


## Stream wiki-18 into shard files  
Only runs if shards do not already exist.


In [9]:
import gzip

def shard_wiki18(gz_path, out_dir, shard_size=100_000):
    banner("SHARDING WIKI-18")
    out_dir.mkdir(parents=True, exist_ok=True)

    existing = list(out_dir.glob("*.jsonl"))
    if existing:
        log.info(f"Shards already exist: {len(existing)} files. Skipping.")
        return

    total = 0
    idx = 0
    written = 0
    out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")

    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except:
                continue

            text = obj.get("text") or obj.get("contents") or ""
            if not text.strip():
                continue

            rec = {
                "id": obj.get("id") or obj.get("page_id") or str(total),
                "contents": text.strip(),
            }
            out_f.write(json.dumps(rec) + "\n")

            total += 1
            written += 1

            if written >= shard_size:
                out_f.close()
                idx += 1
                written = 0
                out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")

    out_f.close()
    log.info(f"Sharding complete. Total docs: {total}")

shard_wiki18(wiki_gz, FULL_SHARDS_DIR)


2025-12-05 06:57:06,912 | INFO | ================================================================================
2025-12-05 06:57:06,912 | INFO | *** SHARDING WIKI-18 ***
2025-12-05 06:57:06,912 | INFO | ================================================================================
2025-12-05 06:57:06,913 | INFO | Shards already exist: 211 files. Skipping.


## Build BM25 index using Pyserini  
Uses Lucene.  
Skips if index already exists.


In [10]:
def build_index(input_dir, index_dir, threads=8):
    banner("BM25 INDEX BUILD")
    if index_dir.exists() and any(index_dir.iterdir()):
        log.info("Index already exists. Skipping.")
        return

    cmd = [
        sys.executable, "-m", "pyserini.index.lucene",
        "--collection", "JsonCollection",
        "--input", str(input_dir),
        "--index", str(index_dir),
        "--generator", "DefaultLuceneDocumentGenerator",
        "--threads", str(threads),
        "--storePositions",
        "--storeDocvectors",
        "--storeRaw"
    ]
    log.info(" ".join(cmd))
    subprocess.run(cmd, check=True)

build_index(FULL_SHARDS_DIR, FULL_INDEX_DIR)


2025-12-05 06:57:07,025 | INFO | ================================================================================
2025-12-05 06:57:07,025 | INFO | *** BM25 INDEX BUILD ***
2025-12-05 06:57:07,026 | INFO | ================================================================================
2025-12-05 06:57:07,026 | INFO | Index already exists. Skipping.


## Sample Queries  
Given a JSONL file with queries (FactScore-Bio etc.), we sample `n_queries=3`  
This cell is idempotent and overwrites only if missing.


In [11]:
def extract_query_text(obj):
    for k in ["query","question","prompt","instruction","text","claim"]:
        if k in obj and isinstance(obj[k], str) and obj[k].strip():
            return obj[k].strip()
    # fallback
    for v in obj.values():
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None


def sample_queries(src_path: Path, out_path: Path, n: int, seed: int):
    banner("LOAD + SAMPLE QUERIES")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        log.info(f"Already exists → {out_path}")
        return out_path

    items = []
    with open(src_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except:
                continue

            q = extract_query_text(obj)
            if not q:
                continue

            qid = str(obj.get("id") or obj.get("_id") or f"s{i:08d}")
            items.append((qid, q))

    random.Random(seed).shuffle(items)
    picked = items[:n]

    with open(out_path, "w", encoding="utf-8") as f:
        for i, (qid, q) in enumerate(picked, 1):
            sid = f"b{i:03d}"
            f.write(json.dumps({"id": sid, "orig_id": qid, "query": q}, ensure_ascii=False) + "\n")

    log.info(f"Wrote {n} sampled queries → {out_path}")
    return out_path


# YOUR SETTINGS
query_src = Path("data/queries/factscore_bio.jsonl")
sampled_queries = Path("data/queries/notebook.seed1337.jsonl")

sample_queries(query_src, sampled_queries, n=50, seed=1337)


2025-12-05 06:57:07,108 | INFO | ================================================================================
2025-12-05 06:57:07,109 | INFO | *** LOAD + SAMPLE QUERIES ***
2025-12-05 06:57:07,109 | INFO | ================================================================================
2025-12-05 06:57:07,110 | INFO | Already exists → data/queries/notebook.seed1337.jsonl


PosixPath('data/queries/notebook.seed1337.jsonl')

In [12]:
!module avail 2>&1 | grep -i java


   ANTLR/2.7.7-GCCcore-10.3.0-Java-11
   ANTLR/2.7.7-GCCcore-11.3.0-Java-11                      (D)
   Java/1.8.0_162
   Java/1.8.0_202                                          (1.8)
   Java/11.0.2                                             (D:11)
   RDP-Classifier/2.13-Java-11
   Trimmomatic/0.39-Java-11
   ant/1.10.11-Java-11


In [13]:
!java -version
!find $JAVA_HOME -name "libjvm.so"


Picked up JAVA_TOOL_OPTIONS: -Xms1g -Xmx8g
openjdk version "21.0.1" 2023-10-17 LTS
OpenJDK Runtime Environment Temurin-21.0.1+12 (build 21.0.1+12-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.1+12 (build 21.0.1+12-LTS, mixed mode, sharing)
/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so


## Retrieval + Reranking  
1. BM25 retrieves top-Kfirst (1000)  
2. CrossEncoder reranks  
3. Keep top-Kkeep (3)  
Everything saved to JSONL.


In [14]:
from pyserini.search.lucene import LuceneSearcher
from sentence_transformers import CrossEncoder

def get_doc_text(raw):
    try:
        obj = json.loads(raw)
        return obj.get("contents", "").strip()
    except:
        return str(raw).strip()


def retrieve_rerank(queries_path: Path,
                    index_dir: Path,
                    out_path: Path,
                    k_first=1000,
                    k_keep=3,
                    ce_model="cross-encoder/ms-marco-MiniLM-L6-v2",
                    batch_size=64):

    banner("BM25 → CE RERANK")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Resume
    done = {}
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    ex = json.loads(line)
                    done[ex["id"]] = ex
                except:
                    pass
        log.info(f"Resume: {len(done)} rows")

    # CrossEncoder
    try:
        ce = CrossEncoder(ce_model, device="cuda")
    except Exception as e:
        log.warning(f"CE unavailable: {e}")
        ce = None

    searcher = LuceneSearcher(str(index_dir))
    searcher.set_bm25(k1=0.9, b=0.4)

    new = 0
    with open(queries_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:

        for line in fin:
            ex = json.loads(line)
            qid, query = ex["id"], ex["query"]

            if qid in done:
                continue

            hits = searcher.search(query, k_first)
            if not hits:
                fout.write(json.dumps({"id": qid, "query": query, "docs": []}) + "\n")
                new += 1
                continue

            candidates = []
            for h in hits:
                raw = searcher.doc(h.docid).raw()
                text = get_doc_text(raw)
                candidates.append({
                    "docid": h.docid,
                    "raw": raw,
                    "bm25": float(h.score),
                    "text": text
                })

            # Rerank
            if ce:
                pairs = [(query, c["text"]) for c in candidates]
                try:
                    scores = ce.predict(pairs, batch_size=batch_size)
                    for c, s in zip(candidates, scores):
                        c["ce"] = float(s)
                    candidates.sort(key=lambda x: x.get("ce", -1e9), reverse=True)
                except Exception as e:
                    log.warning(f"CE failed: {e}")
                    candidates.sort(key=lambda x: x["bm25"], reverse=True)
            else:
                candidates.sort(key=lambda x: x["bm25"], reverse=True)

            final_docs = [
                {"docid": c["docid"], "raw": c["raw"], "bm25": c["bm25"], "ce": c.get("ce")}
                for c in candidates[:k_keep]
            ]

            fout.write(json.dumps({"id": qid, "query": query, "docs": final_docs}) + "\n")
            new += 1
            log.info(f"[{qid}] top docs: {len(final_docs)}")

    log.info(f"Rerank complete — new: {new}")
    return out_path


retr_file = RETR_DIR / "notebook.seed1337.rerank3.jsonl"
retrieve_rerank(sampled_queries, FULL_INDEX_DIR, retr_file, k_first=1000, k_keep=3)


2025-12-05 06:57:10,221 | INFO | 
Using override env var JVM_PATH (/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so) to load libjvm.
Please report your system information (os version, java
version, etc), and the path that works for you, to the
PyJNIus project, at https://github.com/kivy/pyjnius/issues.
so we can improve the automatic discovery.



Picked up JAVA_TOOL_OPTIONS: -Xms1g -Xmx8g


2025-12-05 06:57:10,804 | INFO | Loading faiss with AVX2 support.
2025-12-05 06:57:10,816 | INFO | Successfully loaded faiss with AVX2 support.


/d/hpc/home/ma76193/.local/lib/python3.11/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


2025-12-05 06:57:31,914 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-05 06:57:34,342 | INFO | PyTorch version 2.7.0+cu118 available.
2025-12-05 06:57:34,344 | INFO | Duckdb version 1.3.0 available.
2025-12-05 06:57:36,186 | INFO | ================================================================================
2025-12-05 06:57:36,187 | INFO | *** BM25 → CE RERANK ***
2025-12-05 06:57:36,187 | INFO | ================================================================================
2025-12-05 06:57:36,189 | INFO | Resume: 50 rows
2025-12-05 06:57:37,327 | INFO | Rerank complete — new: 0


Dec 05, 2025 6:57:37 AM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


PosixPath('runs/retrieval/notebook.seed1337.rerank3.jsonl')

In [15]:
import sys
print(sys.executable)
print(sys.version)

import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))


/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/bin/python
3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
True NVIDIA H100 PCIe


## Load Qwen2.5-7B-Instruct (bf16)  
Loads from HF cache if available.  
Used for answer generation with strict "docs-only" policy.


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def parse_docs_for_prompt(docs):
    texts = []
    for i, d in enumerate(docs, 1):
        try:
            obj = json.loads(d["raw"])
            text = obj.get("contents", "")
        except:
            text = d.get("raw", "")
        text = " ".join(text.split())
        if len(text) > 1200:
            text = text[:1195] + " ..."
        texts.append(f"[d{i}] {text}")
    return "\n".join(texts)


def build_prompt(query, docs, strict=True):
    policy = (
        "Answer only using the provided documents. If the documents do not contain the answer, say you don't know."
        if strict else
        "Prefer the provided documents; if insufficient, you may use general knowledge."
    )
    return f"You are a careful assistant. {policy}\n\nQuestion: {query}\n\nDocuments:\n{parse_docs_for_prompt(docs)}\n\nAnswer:"


def load_qwen(model_id="Qwen/Qwen2.5-7B-Instruct"):
    banner("LOAD QWEN 7B (bf16)")
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if torch.cuda.is_available():
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        device = "cuda"
    else:
        mdl = AutoModelForCausalLM.from_pretrained(model_id)
        device = "cpu"

    log.info(f"Loaded Qwen on {device}")
    return tok, mdl, device


qwen_tok, qwen_mdl, qwen_device = load_qwen()


2025-12-05 06:57:37,506 | INFO | ================================================================================
2025-12-05 06:57:37,506 | INFO | *** LOAD QWEN 7B (bf16) ***
2025-12-05 06:57:37,507 | INFO | ================================================================================


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-12-05 06:57:46,325 | INFO | Loaded Qwen on cuda


## Generate Answers (Strict Docs-Only)
Uses Qwen2.5-7B-Instruct (bf16).  
Notebook version supports resume: existing IDs in output JSONL are skipped.


In [17]:
from transformers import TextStreamer

def generate_answers(retr_path: Path,
                     out_path: Path,
                     tok,
                     mdl,
                     device,
                     max_new_tokens=256,
                     strict=True):

    banner("GENERATE ANSWERS — Qwen 7B (bf16)")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done = set()
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except:
                    pass
        log.info(f"Resuming — already have {len(done)} answers.")

    new = 0

    with open(retr_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:

        for line in fin:
            ex = json.loads(line)
            qid = ex["id"]
            q = ex["query"]
            docs = ex["docs"]

            if qid in done:
                continue

            prompt = build_prompt(q, docs, strict=strict)

            enc = tok(prompt, return_tensors="pt")
            if device == "cuda":
                enc = {k: v.cuda() for k, v in enc.items()}

            streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)

            out = mdl.generate(
                **enc,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
                streamer=streamer
            )

            full_text = tok.decode(out[0], skip_special_tokens=True)
            # Extract answer
            if "Answer:" in full_text:
                answer = full_text.split("Answer:", 1)[-1].strip()
            else:
                answer = full_text.strip()

            record = {
                "id": qid,
                "query": q,
                "answer": answer,
                "docs": docs,
                "meta": {
                    "ts": datetime.now().isoformat(timespec="seconds"),
                    "model": "Qwen/Qwen2.5-7B-Instruct",
                    "dtype": "bf16" if device == "cuda" else "fp32",
                    "max_new_tokens": max_new_tokens
                }
            }

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            new += 1

            log.info(f"[{qid}] answer length = {len(answer)}")

    log.info(f"Answer generation complete — new answers: {new}")
    return out_path


answers_file = ANS_DIR / "notebook.seed1337.qwen7b.jsonl"
generate_answers(retr_file, answers_file, qwen_tok, qwen_mdl, qwen_device)


2025-12-05 06:57:46,378 | INFO | ================================================================================
2025-12-05 06:57:46,378 | INFO | *** GENERATE ANSWERS — Qwen 7B (bf16) ***
2025-12-05 06:57:46,378 | INFO | ================================================================================
2025-12-05 06:57:46,380 | INFO | Resuming — already have 50 answers.
2025-12-05 06:57:46,381 | INFO | Answer generation complete — new answers: 0


PosixPath('runs/answers/notebook.seed1337.qwen7b.jsonl')

## SAFE-Lite (Classifier-Only)
Uses `safe-ai/SAFE-Classifier`  
Generates `safe_score` between 0–1  
Higher = more factual confidence.


In [18]:
from pathlib import Path

# This is the file containing your Qwen responses to be evaluated
ans_file = Path("/d/hpc/projects/FRI/ma76193/IR_Project/src/runs/answers/notebook.seed1337.qwen7b.jsonl")

assert ans_file.exists(), f"Answer file not found: {ans_file}"
print("[SAFE] Evaluating responses from:", ans_file)


[SAFE] Evaluating responses from: /d/hpc/projects/FRI/ma76193/IR_Project/src/runs/answers/notebook.seed1337.qwen7b.jsonl


In [19]:
import sys
for k in list(sys.modules.keys()):
    if "TruthTorchLM.long_form_generation.utils" in k:
        del sys.modules[k]
print("Utils cache cleared.")


Utils cache cleared.


In [20]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)


/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/lib/python3.11/site-packages/TruthTorchLM/long_form_generation/utils/safe_utils.py


In [21]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print("Functions:", [f for f in dir(su) if "extract" in f or "strip" in f])


Functions: ['extract_first_code_block', 'extract_first_line', 'extract_first_square_brackets', 'extract_label_from_llm_output', 'strip_string']


In [22]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)


/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/lib/python3.11/site-packages/TruthTorchLM/long_form_generation/utils/safe_utils.py


In [23]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)
print("extract_first_line exists:", hasattr(su, "extract_first_line"))


/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/lib/python3.11/site-packages/TruthTorchLM/long_form_generation/utils/safe_utils.py
extract_first_line exists: True


In [24]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(dir(su))


['Any', 'Literal', 'NO_RESULT_MSG', 'Optional', 'SerperAPI', 'Union', '_SERPER_URL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'clear_line', 'extract_first_code_block', 'extract_first_line', 'extract_first_square_brackets', 'extract_label_from_llm_output', 'maybe_print_error', 'os', 'print_color', 'random', 're', 'requests', 'strip_string', 'termcolor', 'time']


In [25]:
# === SAFE POST-RAG FACT CHECKING WITH QWEN ===========================
print("=== INIT SAFE (QWEN) ===")

import importlib.util
from pathlib import Path
import json
import pandas as pd

# Path to your LOCAL eval_claim.py (already modified to include reranker)
LOCAL_EVAL_CLAIM = Path(
    "/d/hpc/projects/FRI/ma76193/IR_Project/src/TruthTorchLM/src/TruthTorchLM/long_form_generation/evaluators/eval_claim.py"
)

spec = importlib.util.spec_from_file_location("safe_eval_claim", LOCAL_EVAL_CLAIM)
safe_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(safe_module)
ClaimEvaluator = safe_module.ClaimEvaluator

print(f"[SAFE] Loaded ClaimEvaluator from {LOCAL_EVAL_CLAIM}")

# Use already-loaded Qwen model and tokenizer
safe_tok = qwen_tok
safe_mdl = qwen_mdl

print("[SAFE] Using existing Qwen2.5-7B-Instruct model already in memory.")

safe_eval = ClaimEvaluator(
    rater_model=safe_mdl,
    rater_tokenizer=safe_tok,
    lucene_index_dir=str(FULL_INDEX_DIR),
    max_steps=3,
    max_retries=3,
    bm25_k=3,
)

safe_jsonl = UE_DIR / "notebook.seed1337.safe.jsonl"
safe_csv = UE_DIR / "notebook.seed1337.safe.csv"

rows = []

print("=== RUN SAFE ===")

with open(ans_file, "r") as fin, open(safe_jsonl, "w") as fout:
    for line in fin:
        ex = json.loads(line)
        claim = ex["answer"]

        print(f"\n[SAFE] Processing ID={ex['id']}")
        res = safe_eval(claim)

        row = {
            "id": ex["id"],
            "safe_score": res["answer"],
            "safe_response": res["response"],
            "safe_details": res["search_details"],
        }

        rows.append(row)
        fout.write(json.dumps(row) + "\n")

# Save CSV
pd.DataFrame(rows).to_csv(safe_csv, index=False)

print("SAFE completed →", safe_jsonl)
print("SAFE CSV saved →", safe_csv)


=== INIT SAFE (QWEN) ===
[SAFE] Loaded ClaimEvaluator from /d/hpc/projects/FRI/ma76193/IR_Project/src/TruthTorchLM/src/TruthTorchLM/long_form_generation/evaluators/eval_claim.py
[SAFE] Using existing Qwen2.5-7B-Instruct model already in memory.
[SAFE] Loading CrossEncoder reranker: cross-encoder/ms-marco-MiniLM-L6-v2
[SAFE] Lucene index loaded: index/bm25_full
[SAFE] Steps=3, Retries=3, k=3
=== RUN SAFE ===

[SAFE] Processing ID=b001

[SAFE] Evaluating claim: You don't know. 

The provided documents do not contain any information about a  ...
[SAFE] === Step 1/3 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Fernando (footballer, born 1987) joined Galatasaray SK"
[SAFE] Retrieved 1977 characters of evidence
[SAFE] === Step 2/3 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Fernando (footballer, born 1987) joined Galatasaray SK"
[SAFE] Retrieved 1977 characters of evidence
[SAFE] === Step 3/3 ===


...

[SAFE] Query: "Beenie Man Anthony Moses Davis birth date Waterhouse Kingston Jamaica Tastee Talent contest Henry 'Junjo' Lawes Too Fancy"
[SAFE] Retrieved 1961 characters of evidence
[SAFE] === Step 2/3 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Beenie Man Anthony Moses Davis birthplace Waterhouse Kingston Jamaica early career Tastee Talent Henry Junjo Lawes Too Fancy The Invincible Beenie Man"
[SAFE] Retrieved 2028 characters of evidence
[SAFE] === Step 3/3 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Beenie Man Anthony Moses Davis birthplace Waterhouse Kingston Jamaica early career Tastee Talent Henry Junjo Lawes debut single Too Fancy first album Invincible Beenie Man King of Dancehall"
[SAFE] Retrieved 2028 characters of evidence
[SAFE] Final extracted label = Supported
SAFE completed → runs/ue/notebook.seed1337.safe.jsonl
SAFE CSV saved → runs/ue/notebook.seed1337.safe.csv


## MARS (White-box UE)
Notebook version:
- Tries TruthTorchLM (if installed)
- Falls back to NLL(answer|prompt)
All logs printed + saved.


In [26]:
from pathlib import Path
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_mars(answers_file: Path, out_jsonl: Path, out_csv: Path, model_id="Qwen/Qwen2.5-7B-Instruct"):
    print("=== MARS WHITE-BOX ===")

    # Try TruthTorchLM's implementation
    use_tt = False
    try:
        from truthtorchlm import mars as tt_mars
        use_tt = True
        print("Using TruthTorchLM MARS.")
    except Exception:
        print("TruthTorchLM unavailable → fallback.")
        use_tt = False

    # Prepare fallback LM
    tok, mdl = None, None
    if not use_tt:
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
        )

    rows = []

    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)
            qid = ex["id"]
            q = ex["query"]
            ans = ex["answer"]
            docs = ex["docs"]

            score = None
            details = {}

            # 1. TruthTorchLM MARS
            if use_tt:
                try:
                    score, details = tt_mars.compute(
                        question=q,
                        answer=ans,
                        docs=[d.get("raw","") for d in docs],
                        model_name=model_id
                    )
                except Exception as e:
                    print(f"[{qid}] TT error: {e}")

            # 2. Fallback MARS (NLL-based)
            if score is None:
                prompt = build_prompt(q, docs, strict=True)
                full = prompt + "\n\nAnswer: " + ans

                enc = tok(full, return_tensors="pt")
                if torch.cuda.is_available():
                    enc = {k: v.cuda() for k, v in enc.items()}

                # Mask instructions
                prompt_ids = tok(prompt + "\n\nAnswer:", return_tensors="pt")["input_ids"][0]
                full_ids = enc["input_ids"][0]

                labels = full_ids.clone()
                labels[: prompt_ids.size(0)] = -100

                out = mdl(**enc, labels=labels.unsqueeze(0))
                nll = float(out.loss.detach().cpu())
                score = -nll
                details = {"fallback": "nll"}

            rows.append({
                "id": qid,
                "mars_score": score,
                "mars_details": details
            })

            print(f"[{qid}] MARS = {score}")

    # Save JSONL
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    # Save CSV
    pd.DataFrame(rows).to_csv(out_csv, index=False)

    print("MARS done →", out_jsonl)
    print("MARS CSV saved →", out_csv)

    return out_jsonl, out_csv


mars_jsonl = UE_DIR / "notebook.seed1337.mars.jsonl"
mars_csv   = UE_DIR / "notebook.seed1337.mars.csv"

run_mars(answers_file, mars_jsonl, mars_csv)


=== MARS WHITE-BOX ===
TruthTorchLM unavailable → fallback.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[b001] MARS = -0.3670404553413391
[b002] MARS = -0.10150416940450668
[b003] MARS = -0.1773194819688797
[b004] MARS = -0.23079831898212433
[b005] MARS = -0.2482755482196808
[b006] MARS = -0.16151496767997742
[b007] MARS = -0.21184314787387848
[b008] MARS = -0.1907472163438797
[b009] MARS = -0.09796590358018875
[b010] MARS = -0.31747862696647644
[b011] MARS = -0.27702197432518005
[b012] MARS = -0.14034028351306915
[b013] MARS = -0.21308748424053192
[b014] MARS = -0.5475056171417236
[b015] MARS = -0.18973542749881744
[b016] MARS = -0.3398560583591461
[b017] MARS = -0.19965891540050507
[b018] MARS = -0.14555831253528595
[b019] MARS = -0.1799258589744568
[b020] MARS = -0.3671237826347351
[b021] MARS = -0.33156493306159973
[b022] MARS = -0.2657735049724579
[b023] MARS = -0.2961333990097046
[b024] MARS = -0.2979442775249481
[b025] MARS = -0.19217939674854279
[b026] MARS = -0.22812888026237488
[b027] MARS = -0.16538888216018677
[b028] MARS = -0.33134353160858154
[b029] MARS = -0.15423990786075

(PosixPath('runs/ue/notebook.seed1337.mars.jsonl'),
 PosixPath('runs/ue/notebook.seed1337.mars.csv'))

## Eccentricity (Black-box UE)
Embedding-based OOD check.  
Compute:  
- Answer embedding  
- Context centroid  
- Cosine distance  
- Z-score normalization


In [27]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json, csv
from pathlib import Path

def run_ecc(answers_file: Path, jsonl_out: Path, csv_out: Path,
            embed_model="sentence-transformers/all-MiniLM-L6-v2"):

    banner("ECCENTRICITY")

    model = SentenceTransformer(embed_model, device="cuda")

    rows = []

    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)

            qid = ex["id"]
            ans = ex["answer"]
            docs = ex["docs"]

            ctx = [get_doc_text(d.get("raw",""))[:1000] for d in docs if get_doc_text(d.get("raw",""))]

            if not ctx:
                rows.append({"id": qid, "ecc": None, "ecc_z": None})
                continue

            v_ans = model.encode([ans], normalize_embeddings=True)[0]
            v_ctx = model.encode(ctx, normalize_embeddings=True)

            centroid = v_ctx.mean(0)
            cos_sim = float((v_ans * centroid).sum())
            ecc = 1 - cos_sim

            rows.append({"id": qid, "ecc": ecc, "ecc_z": None})

    # compute z-score
    vals = np.array([r["ecc"] for r in rows if r["ecc"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), vals.std(ddof=1) or 1.0
        for r in rows:
            if r["ecc"] is not None:
                r["ecc_z"] = float((r["ecc"] - mu) / sd)

    # save jsonl
    with open(jsonl_out, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    # save csv
    with open(csv_out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "ecc", "ecc_z"])
        w.writeheader()
        for r in rows:
            w.writerow(r)

    return jsonl_out, csv_out


run_ecc(
    answers_file,
    UE_DIR / "factcheck.ecc.jsonl",
    UE_DIR / "factcheck.ecc.csv"
)


2025-12-05 07:00:02,685 | INFO | ================================================================================
2025-12-05 07:00:02,690 | INFO | *** ECCENTRICITY ***
2025-12-05 07:00:02,691 | INFO | ================================================================================
2025-12-05 07:00:02,692 | INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(PosixPath('runs/ue/factcheck.ecc.jsonl'),
 PosixPath('runs/ue/factcheck.ecc.csv'))

In [35]:
from pathlib import Path

# BASE PROJECT ROOT
ROOT = Path("/d/hpc/projects/FRI/ma76193/IR_Project/src")

# Correct folders
QUERIES_DIR = ROOT / "data" / "queries"
ANSWERS_DIR = ROOT / "runs" / "answers"
UE_DIR      = ROOT / "runs" / "ue"

# Correct file names (from your folders)
sampled_queries = QUERIES_DIR / "notebook.seed1337.jsonl"
answers_file    = ANSWERS_DIR / "notebook.seed1337.qwen7b.jsonl"

mars_file = UE_DIR / "notebook.seed1337.mars.jsonl"
ecc_file  = UE_DIR / "notebook.seed1337.ecc.jsonl"
safe_file = UE_DIR / "notebook.seed1337.safe.jsonl"

# Output reports into runs/
REPORTS_DIR = ROOT / "runs"
final_md  = REPORTS_DIR / "notebook.seed1337.report.md"
final_csv = REPORTS_DIR / "notebook.seed1337.scores.csv"
corr_csv  = REPORTS_DIR / "notebook.seed1337.correlations.csv"

print("Queries file: ", sampled_queries.exists(), sampled_queries)
print("Answers file: ", answers_file.exists(), answers_file)
print("MARS file: ", mars_file.exists(), mars_file)
print("ECC file: ", ecc_file.exists(), ecc_file)
print("SAFE file:", safe_file.exists(), safe_file)


Queries file:  True /d/hpc/projects/FRI/ma76193/IR_Project/src/data/queries/notebook.seed1337.jsonl
Answers file:  True /d/hpc/projects/FRI/ma76193/IR_Project/src/runs/answers/notebook.seed1337.qwen7b.jsonl
MARS file:  True /d/hpc/projects/FRI/ma76193/IR_Project/src/runs/ue/notebook.seed1337.mars.jsonl
ECC file:  False /d/hpc/projects/FRI/ma76193/IR_Project/src/runs/ue/notebook.seed1337.ecc.jsonl
SAFE file: True /d/hpc/projects/FRI/ma76193/IR_Project/src/runs/ue/notebook.seed1337.safe.jsonl


## Write Combined CSV + Markdown Report


In [37]:
import json, csv
import pandas as pd
import numpy as np
from pathlib import Path
# Re-declare evaluation output files exactly as your SAFE/MARS/ECC cells produced them


def write_final_report(
    queries_file: Path,
    answers_file: Path,
    mars_file: Path,
    ecc_file: Path,
    safe_file: Path,
    out_md: Path,
    out_csv: Path,
    corr_csv: Path
):
    banner("FINAL REPORT")

    # -----------------------------
    # Load data dictionaries
    # -----------------------------
    qs = {json.loads(line)["id"]: json.loads(line)
          for line in open(queries_file, "r")}

    mars = {json.loads(line)["id"]: json.loads(line)
            for line in open(mars_file, "r")}

    ecc = {json.loads(line)["id"]: json.loads(line)
           for line in open(ecc_file, "r")}

    safe = {json.loads(line)["id"]: json.loads(line)
            for line in open(safe_file, "r")}

    # -----------------------------
    # Build master table in memory
    # -----------------------------
    rows = []

    for line in open(answers_file, "r"):
        ex = json.loads(line)
        qid = ex["id"]

        safe_label = safe.get(qid, {}).get("safe_score")

        # correctness proxy (Supported=1, Not Supported=0)
        if safe_label == "Supported":
            correctness = 1
        elif safe_label == "Not Supported":
            correctness = 0
        else:
            correctness = None

        rows.append({
            "id": qid,
            "query": ex["query"],
            "answer": ex["answer"],
            "mars": mars.get(qid, {}).get("mars_score"),
            "ecc": ecc.get(qid, {}).get("ecc"),
            "ecc_z": ecc.get(qid, {}).get("ecc_z"),
            "safe": safe_label,
            "correctness": correctness,
        })

    df = pd.DataFrame(rows)

    # Save main CSV
    df.to_csv(out_csv, index=False)

    # -----------------------------
    # Save Markdown summary
    # -----------------------------
    with open(out_md, "w") as f:
        f.write("# Notebook RAG Evaluation Report\n\n")
        f.write("| ID | Query | SAFE | MARS | ECC_z | Correctness |\n")
        f.write("|----|-------|------|------|--------|-------------|\n")

        for _, r in df.iterrows():
            q_short = r["query"][:120].replace("|", "/")
            a_safe = r["safe"]
            f.write(f"| {r['id']} | {q_short} | {a_safe} | {r['mars']} | {r['ecc_z']} | {r['correctness']} |\n")

    # -----------------------------
    # CORRELATION ANALYSIS
    # -----------------------------
    corr_df = df[["correctness", "mars", "ecc", "ecc_z"]].dropna()

    correlations = corr_df.corr(method="pearson")
    correlations.to_csv(corr_csv)

    print("\n=== CORRELATION MATRIX (Correctness vs Uncertainty) ===")
    print(correlations)

    banner("DONE")


# -----------------------------
# Paths (same names you used)
# -----------------------------
final_md  = REPORTS_DIR / "notebook.seed1337.report.md"
final_csv = REPORTS_DIR / "notebook.seed1337.scores.csv"
corr_csv  = REPORTS_DIR / "notebook.seed1337.correlations.csv"

write_final_report(
    sampled_queries,
    answers_file,
    mars_file,
    ecc_file,
    safe_json,     # your safe scores file
    final_md,
    final_csv,
    corr_csv
)


2025-12-05 07:07:50,249 | INFO | ================================================================================
2025-12-05 07:07:50,250 | INFO | *** FINAL REPORT ***
2025-12-05 07:07:50,250 | INFO | ================================================================================


FileNotFoundError: [Errno 2] No such file or directory: '/d/hpc/projects/FRI/ma76193/IR_Project/src/runs/ue/notebook.seed1337.ecc.jsonl'

2025-12-05 07:08:01,931 | INFO | [HEARTBEAT] Notebook run alive...
